# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [3]:
!pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms
from tensorboardX import SummaryWriter

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.CIFAR10('cifar10_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.CIFAR10('cifar10_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)



  Using cached tensorboardx-2.6.5-py3-none-any.whl.metadata (6.6 kB)
Using cached tensorboardx-2.6.5-py3-none-any.whl (87 kB)


In [4]:

def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2

def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2

class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std

class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs
    
class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out

class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
        activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512*block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            # layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
    normal='none'):
    #print('initializing PA RN-18 with act {}, normal {}'.format())
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
model.eval()

no input normalization


PreActResNet(
  (normalize): Normalize()
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer1): Sequential(
    (0): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (1): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
  )
  (layer2): Sequential(
    (0): PreActBloc

# Implement the Attacks

Functions are given a simple useful signature that you can start with. Feel free to extend the signature as you see fit.

You may find it useful to create a 'batched' version of PGD that you can use to create the adversarial attack.

In [5]:
def pgd_linf_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        # TODO: Calculate the loss
        loss = ce_loss(output, labels)
        loss.backward()
        # TODO: compute the adv_x
        adv_x = adv_x + eps_step * adv_x.grad.sign()
        # find delta, clamp with eps
        delta = torch.clamp(adv_x - x, min=-eps, max=eps)
        adv_x = torch.clamp(x + delta, min=0, max=1)
        adv_x = adv_x.detach()
   
    return adv_x

In [4]:
def pgd_l2_untargeted(model, x, labels, k, eps, eps_step):
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    adv_x = x.clone().detach()
    adv_x.requires_grad_(True) 
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        batch_size = x.size()[0]
        # TODO: Calculate the loss
        loss = ce_loss(output, labels)
        loss.backward()
        grad = adv_x.grad.data
        # TODO: compute the adv_x
        grad_norms = torch.norm(grad.view(batch_size, -1), p=2, dim=1)
        grad = grad / (grad_norms.view(batch_size, 1, 1, 1) + 1e-10)
        adv_x = adv_x + eps_step * grad
        # find delta, clamp with eps, project delta to the l2 ball
        # HINT: https://github.com/Harry24k/adversarial-attacks-pytorch/blob/master/torchattacks/attacks/pgdl2.py
        delta = adv_x - x
        delta_norms = torch.norm(delta.view(batch_size, -1), p=2, dim=1)
        factor = eps / (delta_norms + 1e-10)
        factor = torch.min(factor, torch.ones_like(delta_norms))
        delta = delta * factor.view(-1, 1, 1, 1)
        adv_x = torch.clamp(x + delta, 0, 1)
        adv_x = adv_x.detach() 
   
    return adv_x

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [27]:
def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            # TODO: get x_adv untargeted pgd linf with eps, and eps_step=eps/4
            x_adv = pgd_linf_untargeted(model, x_batch, y_batch, 10, eps, eps/4)
        elif attack == 'pgd_l2':
            # TODO: get x_adv untargeted pgd l2 with eps, and eps_step=eps/4
            x_adv = pgd_l2_untargeted(model, x_batch, y_batch, 10, eps, eps/4)
        else:
            x_adv = x_batch # standard accuracy
        
        # get the testing accuracy and update tot_test and tot_acc
        predictions = model(x_adv).argmax(dim=1)
        tot_acc += (predictions == y_batch).sum().item()
        tot_test += y_batch.size(0)
        
    if attack == 'pgd_linf' or attack == 'pgd_l2':
        print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on {attack} attack with eps = {eps}')
    else:
        print('Standard accuracy %.5lf' % (tot_acc/tot_test))

In [28]:
# !gdown --folder "https://drive.google.com/drive/folders/1y5E-OSGeS26R8rZAsgUyAbXPx99bXXa_"

Retrieving folder contents
Processing file 1jC8sFpVXpQrmRVxqxhA3skcl768ro_w5 pretr_L2.pth
Processing file 1IjYkXHj5XAF_9U5QW1MlXwiEbRPqUq5H pretr_Linf.pth
Processing file 1XFsvaOzgmEFWABN_dk1UhNv0fBJXOxHf pretr_RAMP.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1jC8sFpVXpQrmRVxqxhA3skcl768ro_w5
From (redirected): https://drive.google.com/uc?id=1jC8sFpVXpQrmRVxqxhA3skcl768ro_w5&confirm=t&uuid=d096921e-91dd-4ad9-b481-43cab7538f97
To: /home/sherryl4/cs521_hw1_models/pretr_L2.pth
100%|██████████████████████████████████████| 44.8M/44.8M [00:00<00:00, 149MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1IjYkXHj5XAF_9U5QW1MlXwiEbRPqUq5H
From (redirected): https://drive.google.com/uc?id=1IjYkXHj5XAF_9U5QW1MlXwiEbRPqUq5H&confirm=t&uuid=406fb4bf-10c5-4c5c-b3ba-ff5fa6b4bc32
To: /home/sherryl4/cs521_hw1_models/pretr_Linf.pth
100%|████████████████████████

## Single-Norm Robust Accuracy

In [38]:
# Evaluate on Linf attack with different models with eps = 8/255
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 8/255
test_model_on_single_attack(model, attack=None, eps=8/255)
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 8/255
test_model_on_single_attack(model, attack=None, eps=8/255)
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 8/255
test_model_on_single_attack(model, attack=None, eps=8/255)
test_model_on_single_attack(model, attack='pgd_linf', eps=8/255)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 74.46it/s]


Standard accuracy 0.82800


Evaluating: 100%|██████████| 157/157 [00:27<00:00,  5.63it/s]


Robust accuracy 0.51210 on pgd_linf attack with eps = 0.03137254901960784


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 72.69it/s]


Standard accuracy 0.88750


... (2 outputs left out)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 73.09it/s]


Standard accuracy 0.81190


Evaluating: 100%|██████████| 157/157 [00:27<00:00,  5.65it/s]

Robust accuracy 0.49750 on pgd_linf attack with eps = 0.03137254901960784


In [40]:
# Evaluate on L2 attack with different models with eps = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on Linf attack with model 1 with eps = 0.75
test_model_on_single_attack(model, attack=None, eps=0.75)
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on Linf attack with model 2 with eps = 0.75
test_model_on_single_attack(model, attack=None, eps=0.75)
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on Linf attack with model 3 with eps = 0.75
test_model_on_single_attack(model, attack=None, eps=0.75)
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 74.46it/s]


Standard accuracy 0.82800


Evaluating: 100%|██████████| 157/157 [00:26<00:00,  5.88it/s]


Robust accuracy 0.48810 on pgd_l2 attack with eps = 0.75


Evaluating: 100%|██████████| 157/157 [00:02<00:00, 73.43it/s]


Standard accuracy 0.88750


... (2 outputs left out)

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 73.18it/s]


Standard accuracy 0.81190


Evaluating: 100%|██████████| 157/157 [00:28<00:00,  5.55it/s]

Robust accuracy 0.59670 on pgd_l2 attack with eps = 0.75


## Multi-Norm Robust Accuracy

In [6]:
def test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # TODO: get x_adv_linf and x_adv_l2 untargeted pgd linf and l2 with eps, and eps_step=eps/4
        x_adv_linf = pgd_linf_untargeted(model, x_batch, y_batch, 10, eps_linf, eps_linf/4)
        x_adv_l2 = pgd_l2_untargeted(model, x_batch, y_batch, 10, eps_l2, eps_l2/4)
        
        ## calculate union accuracy: correct only if both attacks are correct
        
        out = model(x_adv_linf)
        pred_linf = torch.max(out, dim=1)[1]
        out = model(x_adv_l2)
        pred_l2 = torch.max(out, dim=1)[1]

        union = (pred_linf == y_batch) & (pred_l2 == y_batch)
        
        # TODO: get the testing accuracy with multi-norm robustness and update tot_test and tot_acc
        tot_acc += union.sum().item()
        tot_test += y_batch.size(0)
            
    print('Robust accuracy %.5lf' % (tot_acc/tot_test), f'on multi attacks')

In [45]:
# Evaluate on multi-norm attacks with different models with eps_linf = 8./255, eps_l2 = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
# Evaluate on multi attacks with model 1
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
# Evaluate on multi attacks with model 2
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
# Evaluate on multi attacks with model 3
test_model_on_multi_attacks(model, eps_linf=8./255., eps_l2=0.75)

Evaluating: 100%|██████████| 157/157 [00:55<00:00,  2.85it/s]


Robust accuracy 0.47040 on multi attacks


Evaluating: 100%|██████████| 157/157 [00:54<00:00,  2.86it/s]


Robust accuracy 0.30850 on multi attacks


Evaluating: 100%|██████████| 157/157 [00:54<00:00,  2.87it/s]

Robust accuracy 0.49740 on multi attacks


## Homework 3

In [33]:
def train_model(model, eps, num_epochs=10, adv=True):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    for epoch in range(num_epochs):
        running_loss = 0.0

        for i, data in enumerate(train_loader, 0):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            if adv:
                model.eval()
                images = pgd_linf_untargeted(model, images, labels, 10, eps, eps/4).detach()

            model.train()

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

    return model

In [8]:
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
model = train_model(model, eps=0.03, num_epochs=10)

no input normalization
Epoch 1/10, Loss: 1.964
Epoch 2/10, Loss: 1.807
Epoch 3/10, Loss: 1.717
Epoch 4/10, Loss: 1.648
Epoch 5/10, Loss: 1.592
Epoch 6/10, Loss: 1.539
Epoch 7/10, Loss: 1.497
Epoch 8/10, Loss: 1.451
Epoch 9/10, Loss: 1.412
Epoch 10/10, Loss: 1.375


In [29]:
eps_values = np.linspace(0.01, 0.05, 5)
test_model_on_single_attack(model, attack=None, eps=0.01)
for eps in eps_values:
    test_model_on_single_attack(model, attack='pgd_linf', eps=eps)

Evaluating: 100%|██████████| 157/157 [00:01<00:00, 112.13it/s]


Standard accuracy 0.62390


Evaluating: 100%|██████████| 157/157 [00:41<00:00,  3.81it/s]


Robust accuracy 0.53370 on pgd_linf attack with eps = 0.01


Evaluating: 100%|██████████| 157/157 [00:41<00:00,  3.81it/s]


Robust accuracy 0.44600 on pgd_linf attack with eps = 0.02


... (2 outputs left out)

Evaluating: 100%|██████████| 157/157 [00:41<00:00,  3.81it/s]


Robust accuracy 0.27740 on pgd_linf attack with eps = 0.04


Evaluating: 100%|██████████| 157/157 [00:41<00:00,  3.81it/s]

Robust accuracy 0.21460 on pgd_linf attack with eps = 0.05


In [31]:
def test_attacks(model):
    model.eval()
    eps_values = np.linspace(0.01, 0.05, 5)

    for eps in eps_values:
        pgd_correct, fgsm_correct, tot_test = 0.0, 0.0, 0.0

        for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            # PGD 
            x_adv_pgd = pgd_linf_untargeted(model, x_batch, y_batch, 10, eps, eps/4)
            # FGSM (1 step)
            x_adv_fgsm = pgd_linf_untargeted(model, x_batch, y_batch, 1, eps, eps)

            with torch.no_grad():
                predictions_pgd = model(x_adv_pgd).argmax(dim=1)
                predictions_fgsm = model(x_adv_fgsm).argmax(dim=1)

            pgd_correct += (predictions_pgd == y_batch).sum().item()
            fgsm_correct += (predictions_fgsm == y_batch).sum().item()
            tot_test += y_batch.size(0)

        print('PGD robust accuracy %.5lf' % (pgd_correct/tot_test), f'with eps = {eps}')
        print('FGSM robust accuracy %.5lf' % (fgsm_correct/tot_test), f'with eps = {eps}')

In [37]:
original_model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
original_model = train_model(original_model, eps=0.03, num_epochs=10, adv=False)

no input normalization
Epoch 1/10, Loss: 1.338
Epoch 2/10, Loss: 0.875
Epoch 3/10, Loss: 0.690
Epoch 4/10, Loss: 0.576
Epoch 5/10, Loss: 0.496
Epoch 6/10, Loss: 0.429
Epoch 7/10, Loss: 0.367
Epoch 8/10, Loss: 0.320
Epoch 9/10, Loss: 0.271
Epoch 10/10, Loss: 0.229


In [40]:
print("Original model:")
test_model_on_single_attack(original_model, attack=None, eps=0.01)
test_attacks(original_model)

print("Adversarially trained model:")
test_attacks(model)

Original model:


Evaluating: 100%|██████████| 157/157 [00:01<00:00, 112.02it/s]


Standard accuracy 0.76740


Evaluating: 100%|██████████| 157/157 [00:46<00:00,  3.39it/s]


PGD robust accuracy 0.02340 with eps = 0.01
FGSM robust accuracy 0.07140 with eps = 0.01


Evaluating: 100%|██████████| 157/157 [00:46<00:00,  3.39it/s]


... (13 outputs left out)

Evaluating: 100%|██████████| 157/157 [00:46<00:00,  3.39it/s]


PGD robust accuracy 0.27740 with eps = 0.04
FGSM robust accuracy 0.30890 with eps = 0.04


Evaluating: 100%|██████████| 157/157 [00:46<00:00,  3.39it/s]

PGD robust accuracy 0.21460 with eps = 0.05
FGSM robust accuracy 0.25570 with eps = 0.05
